In [14]:
import os
import librosa
import soundfile as sf
import numpy as np
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift, Gain

# -------------------------------
# Parameters
# -------------------------------
input_dir = "/content/drive/MyDrive/Colab Notebooks/Main Project/audio_dataset_padded"          # folder with original 5s, mono audio
output_dir = "/content/drive/MyDrive/Colab Notebooks/Main Project/augmented_audio"         # folder to save augmented audio
os.makedirs(output_dir, exist_ok=True)

sr = 22050          # sample rate
duration = 5.0      # seconds
target_samples_per_class = 250  # goal: 200-300 samples per class

# -------------------------------
# Define augmentation pipeline
# -------------------------------
augmenter = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    Shift(min_shift=-0.1, max_shift=0.1, p=0.5),
    Gain(min_gain_db=-6.0, max_gain_db=6.0, p=0.5),
])

# -------------------------------
# Helper: process original audio
# -------------------------------
def process_audio(file_path):
    y, _ = librosa.load(file_path, sr=sr, mono=True)
    y = librosa.util.fix_length(y, size=int(sr*duration))  # pad/truncate to 5s
    y = y / (np.max(np.abs(y)) + 1e-9)                # normalize
    return y

# -------------------------------
# Augmentation pipeline
# -------------------------------
for class_folder in os.listdir(input_dir):
    class_path = os.path.join(input_dir, class_folder)
    out_class_path = os.path.join(output_dir, class_folder)
    os.makedirs(out_class_path, exist_ok=True)

    files = [f for f in os.listdir(class_path) if f.endswith(".wav")]
    num_files = len(files)

    for file in files:
        file_path = os.path.join(class_path, file)
        y = process_audio(file_path)

        # Save original
        sf.write(os.path.join(out_class_path, file), y, sr)

        # Calculate how many augmentations per clip
        augmentations_per_clip = max(1, int(target_samples_per_class / num_files) - 1)

        # Generate augmented versions
        for i in range(augmentations_per_clip):
            y_aug = augmenter(samples=y, sample_rate=sr)
            base, ext = os.path.splitext(file)
            aug_file = f"{base}_aug{i+1}{ext}"
            sf.write(os.path.join(out_class_path, aug_file), y_aug, sr)

print("✅ Augmentation complete! Dataset ready for training.")

✅ Augmentation complete! Dataset ready for training.


In [12]:
import inspect
from audiomentations import Shift

print(inspect.signature(Shift.__init__))

(self, min_shift: float | int = -0.5, max_shift: float | int = 0.5, shift_unit: Literal['fraction', 'samples', 'seconds'] = 'fraction', rollover: bool = True, fade_duration: float = 0.005, p: float = 0.5)


In [9]:
import inspect
from audiomentations import Gain

print(inspect.signature(Gain.__init__))

(self, min_gain_db: float = -12.0, max_gain_db: float = 12.0, p: float = 0.5)


In [2]:
!pip install audiomentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 19.6 MB/s eta 0:00:00
  Attempting uninstall: soxr
    Found existing installation: soxr 1.0.0
    Uninstalling soxr-1.0.0:
      Successfully uninstalled soxr-1.0.0
